### 베이스라인 데이터 불러오기

In [ ]:
import os
import mlflow
import numpy as np
import pandas as pd
import joblib
from datetime import datetime
from IPython.display import display
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import VotingRegressor, StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

load_dir = os.path.join("D:/seoul_bike/models_pkl/train_pkl")

try:
    X_train = joblib.load(os.path.join(load_dir, "X_train.pkl"))
    Y_train = joblib.load(os.path.join(load_dir, "Y_train.pkl"))
    X_val = joblib.load(os.path.join(load_dir, "X_val.pkl"))
    Y_val = joblib.load(os.path.join(load_dir, "Y_val.pkl"))
    X_test = joblib.load(os.path.join(load_dir, "X_test.pkl"))
    Y_test = joblib.load(os.path.join(load_dir, "Y_test.pkl"))

    print(f"========== 데이터 로드 완료 ==========")
    print(f"학습 데이터 크기: {X_train.shape}")
    print(f"검증 데이터 크기: {X_val.shape}")

except FileNotFoundError as e:
    print(f"파일을 찾을 수 없습니다. 경로를 확인해주세요: {e}")

### 앙상블 베이스라인 (Voting / Stacking)


In [ ]:
# ==========================================\n
# 1. 평가 지표 및 MLflow 설정\n
# ==========================================\n
USE_TEAM_SERVER = True
TEAM_SERVER_URI = "http://223.194.48.21:5000"
AUTHOR = "장수연"

mlflow.set_tracking_uri(TEAM_SERVER_URI if USE_TEAM_SERVER else "sqlite:///mlflow_seoul_bike.db")
mlflow.set_experiment("bike_demand_prediction")

def rmsle(y_true, y_pred):
    log_y = np.log1p(np.maximum(y_true, 0))
    log_pred = np.log1p(np.maximum(y_pred, 0))
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def evaluate_regr(y_true, y_pred):
    return {
        "rmsle": rmsle(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred)
    }

# ==========================================\n
# 2. 베이스 모델 빌더 (병렬 충돌 방지 적용)\n
# ==========================================\n
def build_base_model(model_name, SEED=42, best_params=None):
    """
    사전 튜닝된 하이퍼파라미터(best_params)가 있으면 적용하고,
    없으면 Default 파라미터로 모델을 생성합니다.
    """
    params = best_params if best_params else {}

    # 앙상블 내부에서 병렬처리하므로 개별 모델의 n_jobs는 1로 제한
    if model_name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(**params))
    elif model_name == "LightGBM":
        return LGBMRegressor(random_state=SEED, n_jobs=1, verbosity=-1, **params)
    elif model_name == "XGBoost":
        return XGBRegressor(random_state=SEED, n_jobs=1, tree_method='hist', **params)
    else:
        raise ValueError(f"알 수 없는 모델명: {model_name}")

# ==========================================\n
# 3. 앙상블 학습 루프 (Target Transform 내장)\n
# ==========================================\n
print("\n========== 앙상블 모델 학습 시작 ==========")
ENSEMBLE_BASE_MODELS = ["LightGBM", "XGBoost", "Ridge"]
TARGET_COLUMNS = ['general_rent_cnt', 'sprout_rent_cnt', 'general_rtn_cnt', 'sprout_rtn_cnt']
N_SPLITS = 5
run_date = datetime.now().strftime("%Y-%m-%d %H:%M")

results = []
# TODO: 만약 외부에서 튜닝한 파라미터가 있다면 여기에 로드하세요.
# 예시: best_params_store = joblib.load("tuned_params.pkl")
best_params_store = {}

for target_name in TARGET_COLUMNS:
    y_train_raw = Y_train[target_name]
    y_val_raw = Y_val[target_name]

    # 1. Base Estimators 구성
    base_estimators = []
    for m_name in ENSEMBLE_BASE_MODELS:
        # 튜닝된 파라미터가 없으면 빈 딕셔너리 전달 (에러 방지)
        params = best_params_store.get(target_name, {}).get(m_name, None)
        fresh_model = build_base_model(m_name, best_params=params)
        base_estimators.append((m_name, fresh_model))

    # [피드백 1 반영] Log1p 변환기 셋업 (음수 발생 시 0으로 클리핑)
    def inverse_log_clip(x):
        return np.clip(np.expm1(x), 0, None)

    # ----------------------------------------------------
    # A. Voting Regressor
    # ----------------------------------------------------
    voting_core = VotingRegressor(estimators=base_estimators, n_jobs=-1)

    # 모델 전체를 TransformedTargetRegressor로 감싸서 입력/출력 변환을 자동화
    voting_model = TransformedTargetRegressor(
        regressor=voting_core,
        func=np.log1p,
        inverse_func=inverse_log_clip
    )

    voting_model.fit(X_train, y_train_raw)  # 변환기 덕분에 원본(raw) 타깃을 그대로 넣음
    voting_pred = voting_model.predict(X_val)
    voting_metrics = evaluate_regr(y_val_raw.values, voting_pred)

    with mlflow.start_run(run_name=f"Voting_{target_name}"):
        mlflow.log_param("target", target_name)
        mlflow.log_param("model", "Voting")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(voting_metrics)

    results.append({
        "target": target_name, "model_name": "Voting",
        "RMSLE": voting_metrics["rmsle"], "RMSE": voting_metrics["rmse"], "MAE": voting_metrics["mae"]
    })

    # ----------------------------------------------------
    # B. Stacking Regressor (OOF 기반 누수 차단)
    # ----------------------------------------------------
    stacking_core = StackingRegressor(
        estimators=base_estimators,
        final_estimator=Ridge(),
        cv=TimeSeriesSplit(n_splits=N_SPLITS), # [피드백 4 반영]
        n_jobs=-1
    )

    stacking_model = TransformedTargetRegressor(
        regressor=stacking_core,
        func=np.log1p,
        inverse_func=inverse_log_clip
    )

    stacking_model.fit(X_train, y_train_raw)
    stacking_pred = stacking_model.predict(X_val)
    stacking_metrics = evaluate_regr(y_val_raw.values, stacking_pred)

    with mlflow.start_run(run_name=f"Stacking_{target_name}"):
        mlflow.log_param("target", target_name)
        mlflow.log_param("model", "Stacking")
        mlflow.set_tag("author", AUTHOR)
        mlflow.set_tag("run_date", run_date)
        mlflow.log_metrics(stacking_metrics)

    results.append({
        "target": target_name, "model_name": "Stacking",
        "RMSLE": stacking_metrics["rmsle"], "RMSE": stacking_metrics["rmse"], "MAE": stacking_metrics["mae"]
    })

print("========== 앙상블 모델 학습 완료 ==========")
results_df = pd.DataFrame(results)
display(results_df.sort_values(by=['target', 'RMSLE']))